# Suite TSR — The TypeScript RegExp dialect

ECMA-262's regular expressions (22.2.1, "Patterns"), the sublanguage a regular expression literal holds (12.9.5): the
dialect `Expressions.TypeScript.RegExp` of the TypeScript family, its kinds regexpp's node types, its union, its holder,
its completeness against regexpp's node types kept as data (`grammars/ecmascript-regexpp.json`), its checks, its
editions of ECMA-262, its printing, its evaluation, which refuses each kind by name, and TypeScript5's `constant`, which
holds a literal's parsed regular expression beside its spelling (the dialects proposal, Retrofit's design; mbse-basis
[CAP-226: complete expression dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-226.md)).

In [ ]:
import { readFileSync } from "node:fs";

import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as TypeScriptPatterns from "@mbse/expressions/TypeScript/Patterns";
import * as TypeScriptRegExp from "@mbse/expressions/TypeScript/RegExp";
import * as TypeScriptTypes from "@mbse/expressions/TypeScript/Types";
import { Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, map, raises, same } from "./support.js";

const { NotImplementedError, ValueError } = Errors;
const [TS, TV, TSText] = [TypeScript5.Expressions, TypeScript5.Evaluators, TypeScript5.Text];
const [TSR, TSRText, TST, TSP] = [TypeScriptRegExp.Expressions, TypeScriptRegExp.Text, TypeScriptTypes.Expressions,
  TypeScriptPatterns.Expressions];

const store: any = new Builders.Store(); // the framework, the standard dialects TypeScript5's domains use, and its family
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, Unicode.install, TypeScript5.install,
  TypeScriptTypes.install, TypeScriptPatterns.install, TypeScriptRegExp.install]) {
  store.transaction(install);
}
store.transaction(TypeScriptRegExp.define); // the editions of ECMA-262 the RegExp dialect covers

/** The holder of the term `spec` writes, `dialect`'s own (the RegExp dialect's unless given), written in a
 * transaction. */
function held(spec: unknown, dialect: F.Dialect = TSR.DIALECT): any {
  return store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec))));
}

const regexp = (spec: unknown): any => F.term(held(spec)); // a term of the RegExp dialect, held by its holder
const ts = (spec: unknown): any => F.term(held(spec, TS.DIALECT)); // a term of TypeScript5, held by its holder

/** A spec of the character `text` (or a code point), with its spelling where given. */
function char(text: string | number, raw?: string): (e: any) => any {
  const value = BigInt(typeof text === "number" ? text : text.codePointAt(0)!);
  return (e: any) => e.character((c: any) => (raw === undefined ? c : c.raw(raw)).value(value));
}

/** A spec of an alternative of `elements`. */
const seq = (...elements: unknown[]) => (e: any) => e.alternative((a: any) => a.elements(elements));

/** A spec of a pattern of `alternatives`. */
const pattern = (...alternatives: unknown[]) => (e: any) => e.pattern((p: any) => p.alternatives(alternatives));

// /(?i:a)(?<n>b+?)[a-c\d][\q{xy|z}&&[xy]][\w--a]\p{L}.\b(?=c)\k<n>|$/v: a literal of every kind
const every = (r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq(
  (e: any) => e.group((g: any) => g.modifiers((m: any) => m.modifiers((s: any) => s.add((a: any) => a.ignoreCase(true)))).alternatives([
    seq(char("a"))])),
  (e: any) => e.capturinggroup((g: any) => g.name("n").alternatives([seq((e: any) => e.quantifier((q: any) => q.min(1n).greedy(false).element(
    char("b"))))])),
  (e: any) => e.characterclass((c: any) => c.unicodeSets(true).elements([
    (e: any) => e.characterclassrange((r: any) => r.min(char("a")).max(char("c"))), (e: any) => e.characterset((s: any) => s.setKind("digit"))])),
  (e: any) => e.expressioncharacterclass((x: any) => x.expression((e: any) => e.classintersection((i: any) => i.left(
    (e: any) => e.classstringdisjunction((d: any) => d.alternatives([(e: any) => e.stringalternative((s: any) => s.elements([char("x"), char("y")])),
      (e: any) => e.stringalternative((s: any) => s.elements([char("z")]))]))).right(
    (e: any) => e.characterclass((c: any) => c.unicodeSets(true).elements([char("x"), char("y")])))))),
  (e: any) => e.expressioncharacterclass((x: any) => x.expression((e: any) => e.classsubtraction((s: any) => s.left(
    (e: any) => e.characterset((t: any) => t.setKind("word"))).right(char("a"))))),
  (e: any) => e.characterset((s: any) => s.raw("\\p{L}").setKind("property").strings(false).key("General_Category").value("L")),
  (e: any) => e.characterset((s: any) => s.setKind("any")),
  (e: any) => e.assertion((a: any) => a.assertionKind("word")),
  (e: any) => e.assertion((a: any) => a.assertionKind("lookahead").alternatives([seq(char("c"))])),
  (e: any) => e.backreference((b: any) => b.ref("n"))), seq((e: any) => e.assertion((a: any) => a.assertionKind("end"))))).flags(
  (f: any) => f.unicodeSets(true)));
const EVERY = "/(?i:a)(?<n>b+?)[a-c\\d][\\q{xy|z}&&[xy]][\\w--a]\\p{L}.\\b(?=c)\\k<n>|$/v";

<a id="TSR-01"></a>
## TSR-01 · The RegExp dialect's union, its kinds' and helpers' schemas in regexpp's shapes, the one position that holds it, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
// Its Term: regexpp's node types but its two helpers, and invalid syntax; each kind's and helper's schema is named by
// regexpp's name, under its path
const union: string[] = store.registered(TSR.DIALECT.union).branches.map((b: any) => b.name);
assert(same(union, [...TSR.TAGS, "error", "missing"]) && same(TSR.HELPERS, ["Flags", "ModifierFlags"]));
assert(TSR.DIALECT.name() === "TypeScript RegExp" && TSR.DIALECT.tag === "typescript_regexp");
assert(TSR.DIALECT.path === "TypeScript.RegExp" && TSR.DIALECT.union === "Expressions.TypeScript.RegExp.Term");
const RegExpTerm: any = store.registered(TSR.DIALECT.union);
const kinds = new Map<string, any>(RegExpTerm.branches.map((b: any) => [b.name, b.schema]));
const NAMES = ["RegExpLiteral", "Pattern", "Alternative", "Group", "CapturingGroup", "Quantifier", "CharacterClass",
  "CharacterClassRange", "ClassIntersection", "ClassSubtraction", "ClassStringDisjunction", "StringAlternative",
  "ExpressionCharacterClass", "CharacterSet", "Character", "Assertion", "Backreference", "Modifiers"];
assert(same(TSR.TAGS.map((tag) => kinds.get(tag).name), NAMES.map((name) => `Expressions.TypeScript.RegExp.${name}`)));
assert(same(NAMES.map((name) => name.toLowerCase()), TSR.TAGS)); // each regexpp's node type, lowercased
assert(equal(new Map(TSR.TAGS.map((tag) => [tag, [...kinds.get(tag).properties.keys()]])), map({
  regexpliteral: ["pattern", "flags"], pattern: ["alternatives"], alternative: ["elements"],
  group: ["modifiers", "alternatives"], capturinggroup: ["name", "alternatives"],
  quantifier: ["raw", "min", "max", "greedy", "element"], characterclass: ["unicodeSets", "negate", "elements"],
  characterclassrange: ["min", "max"], classintersection: ["left", "right"], classsubtraction: ["left", "right"],
  classstringdisjunction: ["alternatives"], stringalternative: ["elements"],
  expressioncharacterclass: ["negate", "expression"],
  characterset: ["raw", "setKind", "negate", "strings", "key", "value"], character: ["raw", "value"],
  assertion: ["assertionKind", "negate", "alternatives"], backreference: ["raw", "ref"], modifiers: ["add", "remove"] })));
assert(equal(F._parts(kinds.get("regexpliteral")), [["pattern"], [], ["flags"]]) && equal(F._helpers(kinds.get("regexpliteral")), ["flags"]));
assert(equal(F._parts(kinds.get("group")), [["modifiers"], ["alternatives"], []])); // its modifiers a term, `modifiers`
assert(equal(F._parts(kinds.get("quantifier")), [["element"], [], ["raw", "min", "max", "greedy"]]));
assert(equal(F._parts(kinds.get("modifiers")), [[], [], ["add", "remove"]]) && equal(F._helpers(kinds.get("modifiers")), ["add", "remove"]));
assert(same([...kinds.get("backreference").properties.get("ref").schema.schemas], [store.registered("Schemas.OfInt"),
  store.registered("Schemas.OfStr")])); // a number or a name
// Its helpers, value objects its kinds hold: a literal's flags, a modifiers' flags, each its spelling and its booleans
const helpers = new Map<string, any>(TSR.HELPERS.map((name) => [name, store.registered(`Expressions.TypeScript.RegExp.${name}`)]));
assert(equal(new Map([...helpers].map(([name, h]) => [name, [h.ref, [...h.properties.keys()]]])), map({
  Flags: [false, ["raw", "dotAll", "isGlobal", "hasIndices", "ignoreCase", "multiline", "sticky", "unicode", "unicodeSets"]],
  ModifierFlags: [false, ["raw", "dotAll", "ignoreCase", "multiline"]] })));
assert(kinds.get("regexpliteral").properties.get("flags").schema === helpers.get("Flags"));
assert(kinds.get("modifiers").properties.get("remove").schema === helpers.get("ModifierFlags"));
assert(TSR.FLAGS.map(([, letter]) => letter).join("") === "dgimsuvy" && TSR.MODIFIERS.map(([, letter]) => letter).join("") === "ims");
// None refines Let: a named group's name is a label its backreferences resolve, not a name a term binds
assert(!TSR.TAGS.some((tag) => F.isBinder(kinds.get(tag))));
// Where it is held: TypeScript5's constant, its `regexp`, and nowhere else in the family
const heldAt = [TS.DIALECT, TST.DIALECT, TSP.DIALECT].flatMap((dialect) => store.registered(dialect.union).branches
  .flatMap((b: any) => [...b.schema.properties].filter(([, prop]: any) => prop.schema === RegExpTerm)
    .map(([name]: any) => `${dialect.name()} ${b.name} ${name}`)));
assert(same(heldAt, ["TypeScript5 constant regexp"]));
// Its holder, refining the framework's Expression, its root narrowed to its Term: validated, linked by HasRoot,
// round-tripped
const Holder: any = store.registered("Expressions.TypeScript.RegExp.Expression");
assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
assert(Holder.properties.get("root").schema === RegExpTerm && same([...Holder.adjacencies.keys()], ["abi"]) && Holder.validate().length === 0);
const literal = held(every);
assert(Validators.Validate(store)(Holder, literal).length === 0 && TSR.DIALECT.validate(literal).length === 0);
const linked = store.transaction(() => store.Schemas.OfExpression((s: any) => s.name("tsr.every").expression(
  (a: any) => a.insert((e: any) => e.expression(literal)))));
assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, literal)), [true]));
for (const [encode, decode] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as any[]) {
  assert(F.same(decode(Holder, encode(Holder, literal)).root, literal.root));
  assert(F.same(decode(RegExpTerm, encode(RegExpTerm, F.term(literal))), F.term(literal)));
}
// Its install runs before or after its family's: TypeScript5's names its Term, for a constant's regexp, either creating it
for (const order of [[TypeScriptRegExp.install, TypeScript5.install], [TypeScript5.install, TypeScriptRegExp.install]]) {
  const alone: any = new Builders.Store();
  for (const install of [Framework.install, ...order, TypeScriptTypes.install, TypeScriptPatterns.install]) {
    alone.transaction(install);
  }
  assert(same(alone.registered(TSR.DIALECT.union).branches.map((b: any) => b.name), union));
  const constant = alone.registered(TS.DIALECT.union).branches.find((b: any) => b.name === "constant").schema;
  assert(constant.properties.get("regexp").schema === alone.registered(TSR.DIALECT.union));
}

<a id="TSR-02"></a>
## TSR-02 · It is complete: each of regexpp's node types, kept as data, is a kind or a helper, its properties the schema's, its interfaces' kinds those its checks accept

In [ ]:
interface GrammarNode {
  node: string; kinds?: string[]; helpers?: string[]; interfaces?: Record<string, string[]>;
  properties: Record<string, string>; dropped?: string[];
}
const grammar = globalThis.JSON.parse(readFileSync("../../grammars/ecmascript-regexpp.json", "utf8")) as {
  checked: string; nodes: GrammarNode[] }; // regexpp's node types, as data
assert(grammar.checked.startsWith("@eslint-community/regexpp 4.12.3 (ECMAScript 2025, its latest): its 20 node types"));
const nodes = grammar.nodes;
assert(nodes.length === new Set(nodes.map((node) => node.node)).size && nodes.length === 20); // every node type, once
const mapped = { kinds: [] as string[], helpers: [] as string[] };
const variants = new Map<string, Set<string>>();
for (const node of nodes) { // each a kind or a helper, its properties the schema's, in regexpp's order, renamed where the rule says
  const ways = (["kinds", "helpers"] as const).filter((way) => way in node);
  const way = ways[0]!;
  assert(ways.length === 1 && node[way]!.length === 1 && Object.keys(node).every((key) =>
    ["node", ...ways, "interfaces", "properties", "dropped"].includes(key)), node.node);
  const name = node[way]![0]!;
  mapped[way].push(name);
  let schema: any;
  if (way === "kinds") { // a branch of its Term, its type lowercased
    assert(name === node.node.toLowerCase() && kinds.has(name), node.node);
    schema = kinds.get(name);
  } else { // a value object schema it registers, not a branch
    assert(name === node.node && !kinds.has(name.toLowerCase()), node.node);
    schema = helpers.get(name);
  }
  assert(same([...schema.properties.keys()], Object.values(node.properties)), node.node);
  const renamed = Object.fromEntries(Object.entries(node.properties).filter(([k, v]) => k !== v));
  assert(equal(map(renamed), map(name === "assertion" ? { kind: "assertionKind" } : name === "characterset" ? { kind: "setKind" }
    : name === "Flags" ? { global: "isGlobal" } : {})), node.node); // a keyword of Python or the framework's `kind`
  assert((node.dropped ?? []).every((p) => ["raw", "references", "resolved", "ambiguous"].includes(p)), node.node); // what regexpp derives
  assert(("raw" in node.properties) !== (node.dropped ?? []).includes("raw"), node.node); // every node has its spelling, kept or not
  variants.set(name, new Set(Object.values(node.interfaces ?? {}).flat()));
}
assert(same([...mapped.kinds].sort(), [...TSR.TAGS].sort()) && same(mapped.helpers, TSR.HELPERS));
// A leaf keeps its spelling, and a quantifier its own; every other node is spelled by its parts
assert(same(nodes.filter((node) => "raw" in node.properties).map((node) => node.node), [
  "Quantifier", "Backreference", "Character", "CharacterSet", "Flags", "ModifierFlags"]));
// A type several interfaces share: the kinds of each are the ones its checks accept
assert(equal([...variants.get("assertion")!].sort(), [...TSR.ASSERTIONS].sort()));
assert(equal([...variants.get("characterset")!].sort(), [...TSR.CHARACTERSETS].sort()));
assert(same(nodes.filter((n) => n.interfaces !== undefined).map((n) => n.node), ["CharacterClass", "Assertion", "Backreference", "CharacterSet"]));

<a id="TSR-03"></a>
## TSR-03 · Each kind's checks: ECMA-262's early errors its data can show, and what a kind needs

In [ ]:
const [ca, cz] = [char("a"), char("z")];
const word = (e: any) => e.characterset((s: any) => s.setKind("word"));
for (const [spec, problems] of [
  // a literal: its pattern, and its flags: each once, ECMA-262's, as its spelling spells them, not both u and v
  [(r: any) => r.regexpliteral((l: any) => l), ["a regexpliteral needs a pattern"]],
  [(r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq())).flags((f: any) => f.raw("gg").isGlobal(true))),
    ["a regular expression's flags cannot be 'gg'"]],
  [(r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq())).flags((f: any) => f.raw("g").ignoreCase(true))),
    ["a regular expression's flags 'g' do not spell the flags it holds"]],
  [(r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq())).flags((f: any) => f.unicode(true).unicodeSets(true))),
    ["a regular expression's flags cannot have both u and v"]],
  // a disjunction needs an alternative (an empty one, where it is empty); a group's name, where given, is not empty
  [(r: any) => r.pattern((p: any) => p), ["a pattern needs an alternative"]],
  [(r: any) => r.group((g: any) => g), ["a group needs an alternative"]],
  [(r: any) => r.capturinggroup((g: any) => g.name("")), ["a capturinggroup's name cannot be empty", "a capturinggroup needs an alternative"]],
  [(r: any) => r.classstringdisjunction((d: any) => d), ["a classstringdisjunction needs an alternative"]],
  [(r: any) => r.assertion((s: any) => s.assertionKind("lookbehind")), ["an assertion needs an alternative"]],
  // a quantifier: its element, its bounds in order
  [(r: any) => r.quantifier((q: any) => q.min(0n)), ["a quantifier needs an element"]],
  [(r: any) => r.quantifier((q: any) => q.element(ca)), ["a quantifier needs a min"]],
  [(r: any) => r.quantifier((q: any) => q.min(-1n).element(ca)), ["a quantifier's min must not be negative, got -1"]],
  [(r: any) => r.quantifier((q: any) => q.min(2n).max(1n).element(ca)), ["a quantifier's max 1 is less than its min 2"]],
  [(r: any) => r.quantifier((q: any) => q.min("2").element(ca)), ["a quantifier's min must be a int, got str"]], // its type, by validation
  // a range: of characters, in order
  [(r: any) => r.characterclassrange((c: any) => c.min(cz).max(ca)), ["a characterclassrange's min 122 is greater than its max 97"]],
  [(r: any) => r.characterclassrange((c: any) => c.min(word).max((e: any) => e.error((x: any) => x.text("\\")))),
    ["a characterclassrange's min must be a character, got characterset", "max: invalid syntax '\\\\'"]],
  [(r: any) => r.characterclassrange((c: any) => c.max(ca)), ["a characterclassrange needs a min"]],
  // a class's expression: an intersection or a subtraction
  [(r: any) => r.expressioncharacterclass((x: any) => x.expression(ca)),
    ["an expressioncharacterclass's expression must be a classintersection or a classsubtraction, got character"]],
  [(r: any) => r.expressioncharacterclass((x: any) => x), ["an expressioncharacterclass needs an expression"]],
  // a character set: its kind, a property's key, what each kind has
  [(r: any) => r.characterset((s: any) => s), ["a characterset needs a setKind"]],
  [(r: any) => r.characterset((s: any) => s.setKind("hex")), ["CharacterSet.setKind cannot be 'hex'"]],
  [(r: any) => r.characterset((s: any) => s.setKind("property").value("L")), ["a property characterset needs a key"]],
  [(r: any) => r.characterset((s: any) => s.setKind("property").key("RGI_Emoji").strings(true).negate(true)),
    ["a characterset of strings cannot be negated"]],
  [(r: any) => r.characterset((s: any) => s.setKind("digit").key("L")), ["a digit characterset has no key, value or strings"]],
  [(r: any) => r.characterset((s: any) => s.setKind("any").negate(true)), ["an any characterset cannot be negated"]],
  // a character: a code point
  [(r: any) => r.character((c: any) => c.raw("a")), ["a character needs a value"]],
  [(r: any) => r.character((c: any) => c.value(0x110000n)), ["a character's value must be a code point, got 1114112"]],
  [(r: any) => r.character((c: any) => c.value("a")), ["a character's value must be a int, got str"]], // its type, by validation
  // an assertion: its kind; a lookaround's alternatives, and only a lookaround's or a word boundary's negation
  [(r: any) => r.assertion((s: any) => s), ["an assertion needs an assertionKind"]],
  [(r: any) => r.assertion((s: any) => s.assertionKind("ahead")), ["Assertion.assertionKind cannot be 'ahead'"]],
  [(r: any) => r.assertion((s: any) => s.assertionKind("start").alternatives([seq(ca)])), ["a start assertion holds no alternatives"]],
  [(r: any) => r.assertion((s: any) => s.assertionKind("end").negate(true)), ["an end assertion cannot be negated"]],
  // a backreference: a group's number or its name
  [(r: any) => r.backreference((b: any) => b), ["a backreference needs a ref"]],
  [(r: any) => r.backreference((b: any) => b.ref(0n)), ["a backreference's ref must be a group's number or its name, got 0"]],
  [(r: any) => r.backreference((b: any) => b.ref("")), ["a backreference's ref must be a group's number or its name, got ''"]],
  // modifiers: each of `ims` once, a flag added or removed, none both
  [(r: any) => r.modifiers((m: any) => m.add((f: any) => f.raw("x"))), ["a modifiers' flags cannot be 'x'"]],
  [(r: any) => r.modifiers((m: any) => m.add((f: any) => f.raw(""))), ["a modifiers adds or removes a flag"]],
  [(r: any) => r.modifiers((m: any) => m.add((f: any) => f.ignoreCase(true)).remove((f: any) => f.raw("im").ignoreCase(true).multiline(true))),
    ["a modifiers cannot both add and remove 'i'"]],
  // invalid syntax within a kind, by its container
  [(r: any) => r.alternative((s: any) => s.elements([ca, (e: any) => e.missing((m: any) => m.kind("Atom"))])), ["argument 1: missing Atom"]],
] as [unknown, string[]][]) {
  const found = TSR.DIALECT.validate(regexp(spec));
  assert(same(found, problems), globalThis.JSON.stringify([found, problems]));
}
// What its checks accept: a group's modifiers that remove alone, a lazy quantifier, a range in order, a property of strings
for (const spec of [(r: any) => r.group((g: any) => g.modifiers((m: any) => m.modifiers((s: any) => s.remove((f: any) => f.raw("s").dotAll(true)))).alternatives([seq()])),
  (r: any) => r.quantifier((q: any) => q.min(2n).max(2n).greedy(false).element(ca)),
  (r: any) => r.characterclassrange((c: any) => c.min(ca).max(cz)), (r: any) => r.backreference((b: any) => b.ref(1n)),
  (r: any) => r.characterset((s: any) => s.setKind("property").key("RGI_Emoji").strings(true)),
  (r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq())).flags((f: any) => f.raw("dgimsuy").hasIndices(true).isGlobal(
    true).ignoreCase(true).multiline(true).dotAll(true).unicode(true).sticky(true))), every]) {
  assert(TSR.DIALECT.validate(regexp(spec)).length === 0);
}

<a id="TSR-04"></a>
## TSR-04 · Forms and `make`; it binds nothing and reads its family's dialects, as they read it; evaluation refuses each kind by name

In [ ]:
// Its kinds read and written by their forms, each by the dialect whose union holds it, in its holder: its helpers'
// parts among its attributes, its terms its arguments
const walked = [...F.walk(regexp(every))];
assert(same([...new Set(walked.map((node) => F.kindOf(node)[0]))].sort(), [...TSR.TAGS].sort())); // every kind
for (const node of walked) {
  assert(F.same(F.term(held(TSR.DIALECT.make(F.form(node)))), node));
}
const form = F.form(regexp(every));
const flagsPart = form.attributes.get("flags") as F.Part;
assert(form.kind === "regexpliteral" && same([...form.attributes.keys()], ["flags"]) && flagsPart.kind === "flags");
assert(equal(flagsPart.attributes, map({ unicodeSets: true })) && flagsPart.size === 0);
assert(same(form.arguments.map((argument) => F.kindOf(argument)[0]), ["pattern"]));
const quantifier = walked.find((node) => F.kindOf(node)[0] === "quantifier");
assert(equal(F.form(quantifier).attributes, map({ min: 1n, greedy: false })));
const modifiersForm = F.form(walked.find((node) => F.kindOf(node)[0] === "modifiers"));
const addPart = modifiersForm.attributes.get("add") as F.Part;
assert(modifiersForm.kind === "modifiers" && modifiersForm.attributes.size === 1 && modifiersForm.arguments.length === 0);
assert(addPart.kind === "modifierflags" && equal(addPart.attributes, map({ ignoreCase: true })) && addPart.size === 0);
raises(ValueError, () => TSR.DIALECT.make(new F.Form("call")), "TypeScript RegExp has no kind 'call'"); // neither it nor a
// dialect it reads has one
// It binds nothing: a named group's name is a label, which a backreference resolves; no name is free within it
assert(SY.free(regexp(every), TSR.DIALECT).size === 0 && TSR.DIALECT.validate(regexp((r: any) => r.backreference((b: any) => b.ref("x")))).length === 0);
// It reads its family's other dialects, and each of them reads it; a regular expression's kinds are its own
assert(same(TSR.DIALECT.reads(), [TS.DIALECT, TST.DIALECT, TSP.DIALECT]) && TS.DIALECT.reads().at(-1) === TSR.DIALECT);
assert(TS.DIALECT.kindNamed("character") === TSR.DIALECT.kinds().get("character") && TS.DIALECT.owner(regexp(every)) === TSR.DIALECT);
assert(same([...TSR.DIALECT.kinds().keys()], [...TSR.TAGS, "error", "missing"]));
// Evaluation refuses each kind by its dialect's name, as TypeScript5's evaluator does; inference knows nothing of it
for (const node of walked) {
  raises(NotImplementedError, () => TV.OfAny(node), `TypeScript RegExp ${F.kindOf(node)[0]} is not evaluated`);
}
raises(ValueError, () => TV.OfAny(regexp((r: any) => r.error((e: any) => e.text("(")))), "TypeScript RegExp error is not evaluated");
assert(TSR.DIALECT.infer(regexp(every)).name() === "any");

<a id="TSR-05"></a>
## TSR-05 · Its editions: ECMA-262's, the 5.1 Edition to the 17th, by Ecma International's designations and their common names, each with the kinds it has

In [ ]:
const tsr = store.Instances.Expressions.TypeScript.RegExp;
const DESIGNATIONS = ["ECMA-262 5.1 Edition", ...Array.from({ length: 12 }, (_, i) => `ECMA-262 ${i + 6}th Edition`)];
assert(same([...FS.standards(tsr).keys()], DESIGNATIONS) && same(TSR.EDITIONS.map((edition) => edition[0]), DESIGNATIONS));
assert(equal(FS.properties(FS.find(tsr, "ES5.1")!), map({ bodies: ["org.ecma-international"], number: "262",
  revision: ["version", "5.1"], names: ["ECMAScript 5.1", "ES5.1"] })));
assert(equal(FS.properties(FS.find(tsr, "ECMAScript 2015")!), map({ bodies: ["org.ecma-international"], number: "262",
  revision: ["edition", 6n], names: ["ECMAScript 2015", "ES2015"] })));
assert(same(FS.tags(FS.find(tsr, "ES2025")!), ["ECMA-262 16th Edition", "ECMAScript 2025", "ES2025"])); // found by any of its tags
assert(same(FS.tags(FS.find(tsr, "ES2026")!), ["ECMA-262 17th Edition", "ECMAScript 2026", "ES2026"])); // its pattern grammar the 16th's
assert(FS.find(tsr, "ES2027") === null && FS.find(tsr, "TypeScript 5.0") === null); // its own editions alone
// Each edition its kinds: every one but the `v` flag's five, from the 15th (ES2024), and modifiers, from the 16th (ES2025);
// the 17th (ES2026) adds none
const V = ["classintersection", "classsubtraction", "classstringdisjunction", "stringalternative", "expressioncharacterclass"];
assert(equal(TSR.SINCE, new Map([...V.map((tag): [string, number] => [tag, 10]), ["modifiers", 11]])));
const counts = [...FS.standards(tsr).values()].map((standard) => FS.kinds(standard).length);
assert(same(counts, [...Array(10).fill(12), 17, 18, 18]));
assert(same(FS.kinds(FS.find(tsr, "ES2026")!).map((s: any) => s.name), TSR.TAGS.map((tag) => kinds.get(tag).name))); // its own kinds alone
assert(same([...FS.of(tsr, kinds.get("modifiers")).values()].map((s) => FS.tags(s).at(-1)), ["ES2025", "ES2026"]));
assert(same([...FS.of(tsr, kinds.get("classintersection")).values()].map((s) => FS.tags(s).at(-1)), ["ES2024", "ES2025", "ES2026"]));
assert(FS.of(tsr, kinds.get("character")).size === 13 && FS.of(tsr, store.registered("Expressions.Framework.Error")).size === 0);

<a id="TSR-06"></a>
## TSR-06 · Printed as ECMAScript writes a regular expression, by the family's printer: every kind, a spelling as written, else its default spelling

In [ ]:
assert(TSRText.ToText === TSText.ToText && same([...TSRText.TEXTS.keys()].sort(), [...TSR.TAGS].sort()));
assert(TSText.ToText(held(every)) === EVERY); // every kind, from its own holder
const group = (m: unknown) => (r: any) => r.group((g: any) => g.modifiers((e: any) => e.modifiers(m)).alternatives([seq(char("a"))]));
const quantified = (q: (s: any) => any) => (r: any) => r.quantifier((s: any) => q(s).element(char("a")));
const inClass = (...elements: unknown[]) => (r: any) => r.characterclass((c: any) => c.elements(elements));
for (const [spec, text] of [
  // a spelling as written: a character's, a character set's, a backreference's, a literal's and a modifiers' flags, a
  // quantifier's where it begins with its element's text
  [char("A", "\\x41"), "\\x41"], [(r: any) => r.characterset((s: any) => s.raw("\\p{gc=L}").setKind("property").key("gc").value("L")), "\\p{gc=L}"],
  [(r: any) => r.backreference((b: any) => b.raw("\\k<n>").ref("n")), "\\k<n>"],
  [(r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq(char("a")))).flags((f: any) => f.raw("ig").isGlobal(true).ignoreCase(true))), "/a/ig"],
  [group((s: any) => s.add((f: any) => f.raw("mi").ignoreCase(true).multiline(true))), "(?mi:a)"],
  [quantified((q) => q.raw("a{0,}").min(0n)), "a{0,}"], [quantified((q) => q.raw("b{0,}").min(0n)), "a*"],
  // else its default spelling: a quantifier's shortest, lazy where it is not greedy
  [quantified((q) => q.min(1n)), "a+"], [quantified((q) => q.min(0n).max(1n)), "a?"],
  [quantified((q) => q.min(2n)), "a{2,}"], [quantified((q) => q.min(3n).max(3n)), "a{3}"],
  [quantified((q) => q.min(2n).max(5n).greedy(false)), "a{2,5}?"], [quantified((q) => q.min(0n).greedy(false)), "a*?"],
  // a character: itself, a syntax character and `/` escaped, in a class `-` too, a control or a line terminator escaped
  [(r: any) => r.alternative((s: any) => s.elements([...[..."^$\\.*+?()[]{}|/-é"].map((c) => char(c)), char(0x1F600)])),
    "\\^\\$\\\\\\.\\*\\+\\?\\(\\)\\[\\]\\{\\}\\|\\/-é😀"],
  [inClass(char("-"), char("]"), char("^"), (e: any) => e.characterclassrange((c: any) => c.min(char("-")).max(char("/")))),
    "[\\-\\]\\^\\--\\/]"],
  [(r: any) => r.alternative((s: any) => s.elements([9, 10, 11, 12, 13, 0, 0x1F, 0x7F, 0x2028, 0x2029, 0xD800].map((c) => char(c)))),
    "\\t\\n\\v\\f\\r\\x00\\x1F\\x7F\\u2028\\u2029\\uD800"],
  // a character set, an assertion, a group, a backreference, a class, a string disjunction, flags
  [(r: any) => r.alternative((s: any) => s.elements(["digit", "space", "word"].flatMap((kind) => [false, true].map(
    (negate) => (e: any) => e.characterset((c: any) => c.setKind(kind).negate(negate)))))), "\\d\\D\\s\\S\\w\\W"],
  [(r: any) => r.characterset((s: any) => s.setKind("property").key("Script").value("Greek").negate(true)), "\\P{Script=Greek}"],
  [(r: any) => r.characterset((s: any) => s.setKind("property").key("RGI_Emoji").strings(true)), "\\p{RGI_Emoji}"],
  [(r: any) => r.alternative((s: any) => s.elements(([["start", false], ["end", false], ["word", false], ["word", true],
    ["lookahead", false], ["lookahead", true], ["lookbehind", false], ["lookbehind", true]] as [string, boolean][]).map(([kind, negate]) =>
    (e: any) => e.assertion((a: any) => a.assertionKind(kind).negate(negate).alternatives(kind.startsWith("look") ? [seq(char("a"))] : []))))),
    "^$\\b\\B(?=a)(?!a)(?<=a)(?<!a)"],
  [(r: any) => r.group((g: any) => g.alternatives([seq(char("a")), seq()])), "(?:a|)"],
  [group((s: any) => s.remove((f: any) => f.multiline(true))), "(?-m:a)"],
  [group((s: any) => s.add((f: any) => f.ignoreCase(true).dotAll(true)).remove((f: any) => f.multiline(true))), "(?is-m:a)"],
  [(r: any) => r.capturinggroup((g: any) => g.alternatives([seq(char("a"))])), "(a)"],
  [(r: any) => r.backreference((b: any) => b.ref(12n)), "\\12"],
  [(r: any) => r.characterclass((c: any) => c.negate(true).elements([char("a"), char("A", "\\x41")])), "[^a\\x41]"],
  [(r: any) => r.expressioncharacterclass((x: any) => x.negate(true).expression((e: any) => e.classsubtraction((s: any) => s.left(
    char("-")).right((e: any) => e.classstringdisjunction((d: any) => d.alternatives([(e: any) => e.stringalternative(
      (s: any) => s.elements([char("|"), char("-")]))])))))), "[^\\---\\q{\\|\\-}]"],
  [(r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq(char("/")))).flags((f: any) => f.sticky(true).hasIndices(true).isGlobal(
    true).ignoreCase(true).multiline(true).dotAll(true).unicodeSets(true))), "/\\//dgimsvy"],
  [(r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq()))), "//"], // no flags; an empty pattern, as it holds it
  // invalid syntax, as written
  [(r: any) => r.alternative((s: any) => s.elements([char("a"), (e: any) => e.error((x: any) => x.text("(?")),
    (e: any) => e.missing((m: any) => m.kind("Atom"))])), "a(?"],
] as [unknown, string][]) {
  assert(TSText.ToText(regexp(spec)) === text, globalThis.JSON.stringify([TSText.ToText(regexp(spec)), text]));
}
raises(ValueError, () => TSText.ToText(regexp((r: any) => r.error((e: any) => e))), "TypeScript RegExp error has no text to print"); // by
// its dialect's name

<a id="TSR-07"></a>
## TSR-07 · TypeScript5's `constant` holds a literal's regular expression, its `regexp`, beside its spelling: validated with it, printed from it where it has no spelling, read and written by its form

In [ ]:
// Its schema: its spelling, `pattern` and `flags`, then `regexp`, the regular expression they encode, a slot holding the
// RegExp dialect's Term
const Constant = store.registered(TS.DIALECT.union).branches.find((b: any) => b.name === "constant").schema;
assert(same([...Constant.properties.keys()], ["singleton", "pattern", "flags", "regexp", "text", "problems"]));
assert(Constant.properties.get("regexp").schema === RegExpTerm);
assert(equal(F._parts(Constant), [["regexp"], [], ["singleton", "pattern", "flags", "text", "problems"]]));
const slash = (r: any) => r.regexpliteral((l: any) => l.pattern(pattern(seq(char("a")))).flags((f: any) => f.raw("g").isGlobal(true)));
const spelled = ts((e: any) => e.constant((c: any) => c.pattern("a").flags("g").regexp(slash))); // /a/g, read by a parser
assert(TS.DIALECT.validate(spelled).length === 0 && TSText.ToText(spelled) === "/a/g");
assert(equal(F.form(spelled).attributes, map({ pattern: "a", flags: "g" })) && same(F.form(spelled).arguments.map((a) => F.kindOf(a)[0]), ["regexpliteral"]));
assert(same([...F.walk(spelled)].map((node) => F.kindOf(node)[0]), ["constant", "regexpliteral", "pattern", "alternative", "character"]));
assert(F.same(F.term(held(TS.DIALECT.make(F.form(spelled)), TS.DIALECT)), spelled));
assert(SY.free(spelled, TS.DIALECT).size === 0 && TS.DIALECT.infer(spelled).name() === "RegExp");
const whole = store.registered("Expressions.TypeScript.TypeScript5.Expression"); // round-tripped in its holder
for (const [encode, decode] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as any[]) {
  assert(F.same(decode(whole, encode(whole, F.holderOf(spelled))).root, spelled));
}
// A constant written with its regexp alone, as code writes one: printed from it
const written = ts((e: any) => e.constant((c: any) => c.regexp(slash)));
assert(TS.DIALECT.validate(written).length === 0 && TSText.ToText(written) === "/a/g");
// Its text (`constantText`), as the printer writes it, from its regexp's text, which only the printer gives
assert(TS.constantText(F.form(spelled).attributes) === "/a/g" && TS.constantText(new Map(), () => "/a/g") === "/a/g");
raises(ValueError, () => TS.constantText(F.form(written).attributes),
  "a TypeScript5 constant with no pattern is written from its regexp, by its printer");
assert(TSText.ToText(ts((e: any) => e.constant((c: any) => c.regexp(every)))) === EVERY);
// Its checks: its regexp's problems too, by its own dialect; a singleton holds none
for (const [spec, problems] of [
  [(c: any) => c.pattern("a(").regexp((r: any) => r.regexpliteral((l: any) => l)), ["regexp: a regexpliteral needs a pattern"]],
  [(c: any) => c.singleton("null").regexp(slash), ["a constant holds a singleton or a regular expression, not both"]],
  [(c: any) => c.flags("g"), ["a regular expression needs a pattern"]],
  [(c: any) => c.pattern("a").flags("gg"), ["a regular expression's flags cannot be 'gg'"]],
] as [unknown, string[]][]) {
  assert(same(TS.DIALECT.validate(ts((e: any) => e.constant(spec))), problems));
}
// Evaluation refuses a constant, as TypeScript5's evaluator does any kind but a literal; a string given to `new RegExp`
// is a string, not a term of this dialect
raises(NotImplementedError, () => TV.OfAny(spelled), "TypeScript5 constant is not evaluated");
const built = ts((e: any) => e.newexpression((n: any) => n.callee((c: any) => c.identifier((i: any) => i.name("RegExp"))).arguments([
  (a: any) => a.literal((li: any) => li.value((v: any) => v.as_scalar("a+")))])));
assert(same([...new Set([...F.walk(built)].map((node) => F.dialectOf(node)))], [TS.DIALECT.union]));